In [1]:
import os
import glob
import json
import numpy as np
import pandas as pd
import faiss

from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer
from openai import OpenAI

load_dotenv()

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")

print("API Key 是否读取成功：", DEEPSEEK_API_KEY is not None)

API Key 是否读取成功： True


In [2]:
DATA_DIR = "data"

txt_files = glob.glob(os.path.join(DATA_DIR, "*.txt"))

print("读取到的 txt 文件：")
for file in txt_files:
    print(file)

docs = []

for file_path in txt_files:
    with open(file_path, "r", encoding="utf-8") as f:
        text = f.read()
    
    docs.append({
        "source": os.path.basename(file_path),
        "text": text
    })

print("\n文档数量：", len(docs))

if docs:
    print("\n第一篇文档预览：")
    print(docs[0]["text"][:500])
else:
    print("没有读取到文档，请检查 data 文件夹。")

读取到的 txt 文件：
data\01_网络安全基础概念.txt
data\02_常见网络攻击类型.txt
data\03_安全防护与最佳实践.txt

文档数量： 3

第一篇文档预览：
网络安全基础概念

网络安全是指保护网络系统中的硬件、软件及数据资源，防止因偶然或恶意原因遭到破坏、更改或泄露，确保系统连续可靠地运行。

网络安全的核心目标遵循CIA三元组原则：

1. 机密性：确保信息不泄露给非授权的个人或实体。常见的实现技术包括数据加密、访问控制列表等。例如，在传输用户密码时使用HTTPS协议进行加密，就是保障机密性的典型应用。

2. 完整性：保护信息不被未授权的篡改或破坏。哈希校验和数字签名是主要技术手段。当你从官方网站下载软件时，通常会提供一个MD5或SHA-256校验值，用户可以据此验证文件是否被植入恶意代码。

3. 可用性：保证被授权实体在需要时可以访问和使用相关资源。这通常涉及冗余设计、DDoS攻击防御和灾难恢复计划。一个典型的例子是银行系统必须保证用户能够全天候进行交易查询。

除了CIA三原则，可审计性和不可抵赖性也是现代网络安全的重要延伸属性。防火墙作为第一道防线，按照既定规则过滤进出网络的数据包，是网络安全架构中最基础的设施之一。


In [3]:
def clean_text(text):
    """
    基础文本清洗：
    1. 统一换行符；
    2. 合并过多空行；
    3. 去掉多余空格；
    4. 尽量保留段落结构。
    """
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    return text.strip()


def split_long_paragraph(paragraph, chunk_size=400, overlap=80):
    """
    如果单个段落太长，则使用滑动窗口继续切分。
    """
    chunks = []
    start = 0
    
    while start < len(paragraph):
        end = start + chunk_size
        chunk = paragraph[start:end].strip()
        
        if chunk:
            chunks.append(chunk)
        
        if end >= len(paragraph):
            break
        
        start += chunk_size - overlap
    
    return chunks


def split_text_by_paragraph(text, chunk_size=400, overlap=80, min_chunk_size=80):
    """
    段落优先的中文文档切分方法：
    1. 先按空行切段落；
    2. 尽量合并短段落；
    3. 太长段落再滑动窗口切分；
    4. 避免产生太短的 chunk。
    """
    text = clean_text(text)
    
    paragraphs = [p.strip() for p in text.split("\n\n") if p.strip()]
    
    chunks = []
    current_chunk = ""
    
    for para in paragraphs:
        # 如果段落本身太长，先保存当前 chunk，再切长段落
        if len(para) > chunk_size:
            if len(current_chunk) >= min_chunk_size:
                chunks.append(current_chunk.strip())
                current_chunk = ""
            
            long_chunks = split_long_paragraph(
                para,
                chunk_size=chunk_size,
                overlap=overlap
            )
            chunks.extend(long_chunks)
            continue
        
        # 如果当前 chunk 加上新段落不超长，则合并
        if len(current_chunk) + len(para) + 2 <= chunk_size:
            if current_chunk:
                current_chunk += "\n" + para
            else:
                current_chunk = para
        else:
            # 保存当前 chunk
            if len(current_chunk) >= min_chunk_size:
                chunks.append(current_chunk.strip())
            
            current_chunk = para
    
    # 保存最后一个 chunk
    if len(current_chunk) >= min_chunk_size:
        chunks.append(current_chunk.strip())
    
    return chunks

In [4]:
import re

all_chunks = []

for doc in docs:
    chunks = split_text_by_paragraph(
        doc["text"],
        chunk_size=400,
        overlap=80,
        min_chunk_size=80
    )
    
    for i, chunk in enumerate(chunks):
        all_chunks.append({
            "source": doc["source"],
            "chunk_id": i,
            "text": chunk,
            "chunk_length": len(chunk)
        })

print("文档数量：", len(docs))
print("总 chunk 数：", len(all_chunks))

print("\n前 3 个 chunk 示例：")
for item in all_chunks[:3]:
    print("=" * 80)
    print("source:", item["source"])
    print("chunk_id:", item["chunk_id"])
    print("chunk_length:", item["chunk_length"])
    print(item["text"][:300])

文档数量： 3
总 chunk 数： 4

前 3 个 chunk 示例：
source: 01_网络安全基础概念.txt
chunk_id: 0
chunk_length: 357
网络安全基础概念
网络安全是指保护网络系统中的硬件、软件及数据资源，防止因偶然或恶意原因遭到破坏、更改或泄露，确保系统连续可靠地运行。
网络安全的核心目标遵循CIA三元组原则：
1. 机密性：确保信息不泄露给非授权的个人或实体。常见的实现技术包括数据加密、访问控制列表等。例如，在传输用户密码时使用HTTPS协议进行加密，就是保障机密性的典型应用。
2. 完整性：保护信息不被未授权的篡改或破坏。哈希校验和数字签名是主要技术手段。当你从官方网站下载软件时，通常会提供一个MD5或SHA-256校验值，用户可以据此验证文件是否被植入恶意代码。
3. 可用性：保证被授权实体在需要时可以访问和使用相关资源
source: 01_网络安全基础概念.txt
chunk_id: 1
chunk_length: 80
除了CIA三原则，可审计性和不可抵赖性也是现代网络安全的重要延伸属性。防火墙作为第一道防线，按照既定规则过滤进出网络的数据包，是网络安全架构中最基础的设施之一。
source: 02_常见网络攻击类型.txt
chunk_id: 0
chunk_length: 378
常见网络攻击类型
在网络安全领域，了解攻击者的手段是构建防御体系的基础。社会工程学是最具欺骗性的攻击方式，攻击者利用人性弱点如好奇心或恐惧心理，伪装成合法实体诱骗受害者泄露敏感信息。典型的钓鱼邮件会伪造发件人地址，诱导用户点击恶意链接。
分布式拒绝服务攻击旨在耗尽目标服务器资源，导致其无法为正常用户提供服务。攻击者通常控制大量被感染的“僵尸”计算机，同时向目标发起海量请求，造成带宽拥堵或系统崩溃。
中间人攻击发生在通信双方不知情的情况下，攻击者秘密中继并可能篡改双方的通信内容。在不安全的公共Wi-Fi环境中，攻击者可以轻易实施此攻击，窃取用户的登录凭证。
SQL注入攻击针对数据库驱动的应用程序


In [5]:
embedding_model_name = "paraphrase-multilingual-MiniLM-L12-v2"

embedder = SentenceTransformer(embedding_model_name)

print("Embedding 模型加载完成：", embedding_model_name)

Embedding 模型加载完成： paraphrase-multilingual-MiniLM-L12-v2


In [6]:
texts = [item["text"] for item in all_chunks]

embeddings = embedder.encode(
    texts,
    normalize_embeddings=True,
    show_progress_bar=True
)

embeddings = np.array(embeddings).astype("float32")

print("向量矩阵形状：", embeddings.shape)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

向量矩阵形状： (4, 384)


In [7]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)
index.add(embeddings)

print("FAISS 索引建立完成")
print("索引中的向量数量：", index.ntotal)

FAISS 索引建立完成
索引中的向量数量： 4


In [8]:
def retrieve(query, top_k=3):
    """
    输入用户问题，返回 Top-K 相似文本块。
    """
    query_embedding = embedder.encode(
        [query],
        normalize_embeddings=True
    )
    query_embedding = np.array(query_embedding).astype("float32")
    
    scores, indices = index.search(query_embedding, top_k)
    
    results = []
    
    for score, idx in zip(scores[0], indices[0]):
        item = all_chunks[idx]
        results.append({
            "score": float(score),
            "source": item["source"],
            "chunk_id": item["chunk_id"],
            "chunk_length": item.get("chunk_length", len(item["text"])),
            "text": item["text"]
        })
    
    return results


def show_retrieved_docs(retrieved_docs, max_chars=300):
    """
    展示检索结果，方便判断检索是否准确。
    """
    print("\n========== 检索结果 ==========")
    
    for i, doc in enumerate(retrieved_docs, start=1):
        print(f"\nTop {i}")
        print(f"score: {doc['score']:.4f}")
        print(f"source: {doc['source']}")
        print(f"chunk_id: {doc['chunk_id']}")
        print(f"chunk_length: {doc['chunk_length']}")
        print("text:")
        print(doc["text"][:max_chars])
        print("-" * 80)

In [9]:
query = "CIA三元组原则有哪些？"

retrieved_docs = retrieve(query, top_k=3)
show_retrieved_docs(retrieved_docs)


========== 检索结果 ==========

Top 1
score: 0.6699
source: 01_网络安全基础概念.txt
chunk_id: 1
chunk_length: 80
text:
除了CIA三原则，可审计性和不可抵赖性也是现代网络安全的重要延伸属性。防火墙作为第一道防线，按照既定规则过滤进出网络的数据包，是网络安全架构中最基础的设施之一。
--------------------------------------------------------------------------------

Top 2
score: 0.5760
source: 01_网络安全基础概念.txt
chunk_id: 0
chunk_length: 357
text:
网络安全基础概念
网络安全是指保护网络系统中的硬件、软件及数据资源，防止因偶然或恶意原因遭到破坏、更改或泄露，确保系统连续可靠地运行。
网络安全的核心目标遵循CIA三元组原则：
1. 机密性：确保信息不泄露给非授权的个人或实体。常见的实现技术包括数据加密、访问控制列表等。例如，在传输用户密码时使用HTTPS协议进行加密，就是保障机密性的典型应用。
2. 完整性：保护信息不被未授权的篡改或破坏。哈希校验和数字签名是主要技术手段。当你从官方网站下载软件时，通常会提供一个MD5或SHA-256校验值，用户可以据此验证文件是否被植入恶意代码。
3. 可用性：保证被授权实体在需要时可以访问和使用相关资源
--------------------------------------------------------------------------------

Top 3
score: 0.5697
source: 03_安全防护与最佳实践.txt
chunk_id: 0
chunk_length: 359
text:
网络安全防护与最佳实践
构建有效的网络安全防护体系需要遵循纵深防御原则，即通过部署多层安全控制措施，确保单一防线被突破后仍有其他机制提供保护。
身份认证是安全的第一道关卡。仅依赖密码已不足以应对现代威胁，启用多因素认证可显著提升账户安全性。它通过组合“你知道的”、“你拥有的”和“你是什么的”三类因素，有效防止凭证泄露后的未授权访问。
数据加密是保护信息机密性的基石，分

In [10]:
def build_prompt(query, retrieved_docs):
    context = "\n\n".join([
        f"[来源: {doc['source']} | chunk: {doc['chunk_id']} | score: {doc['score']:.4f}]\n{doc['text']}"
        for doc in retrieved_docs
    ])
    
    prompt = f"""
你是一个严谨的网络安全知识库问答助手。你只能根据【检索资料】回答问题。

请遵守以下规则：
1. 只使用【检索资料】中的信息回答；
2. 如果资料不足以回答，请直接说“知识库中没有足够信息回答该问题”；
3. 不要补充资料中没有出现的事实、案例、工具或数据；
4. 回答要简洁、分点；
5. 最后列出你参考的来源文件名和 chunk_id。

【检索资料】
{context}

【用户问题】
{query}

【回答】
"""
    return prompt

In [11]:
prompt = build_prompt(query, retrieved_docs)
print(prompt[:1500])


你是一个严谨的网络安全知识库问答助手。你只能根据【检索资料】回答问题。

请遵守以下规则：
1. 只使用【检索资料】中的信息回答；
2. 如果资料不足以回答，请直接说“知识库中没有足够信息回答该问题”；
3. 不要补充资料中没有出现的事实、案例、工具或数据；
4. 回答要简洁、分点；
5. 最后列出你参考的来源文件名和 chunk_id。

【检索资料】
[来源: 01_网络安全基础概念.txt | chunk: 1 | score: 0.6699]
除了CIA三原则，可审计性和不可抵赖性也是现代网络安全的重要延伸属性。防火墙作为第一道防线，按照既定规则过滤进出网络的数据包，是网络安全架构中最基础的设施之一。

[来源: 01_网络安全基础概念.txt | chunk: 0 | score: 0.5760]
网络安全基础概念
网络安全是指保护网络系统中的硬件、软件及数据资源，防止因偶然或恶意原因遭到破坏、更改或泄露，确保系统连续可靠地运行。
网络安全的核心目标遵循CIA三元组原则：
1. 机密性：确保信息不泄露给非授权的个人或实体。常见的实现技术包括数据加密、访问控制列表等。例如，在传输用户密码时使用HTTPS协议进行加密，就是保障机密性的典型应用。
2. 完整性：保护信息不被未授权的篡改或破坏。哈希校验和数字签名是主要技术手段。当你从官方网站下载软件时，通常会提供一个MD5或SHA-256校验值，用户可以据此验证文件是否被植入恶意代码。
3. 可用性：保证被授权实体在需要时可以访问和使用相关资源。这通常涉及冗余设计、DDoS攻击防御和灾难恢复计划。一个典型的例子是银行系统必须保证用户能够全天候进行交易查询。

[来源: 03_安全防护与最佳实践.txt | chunk: 0 | score: 0.5697]
网络安全防护与最佳实践
构建有效的网络安全防护体系需要遵循纵深防御原则，即通过部署多层安全控制措施，确保单一防线被突破后仍有其他机制提供保护。
身份认证是安全的第一道关卡。仅依赖密码已不足以应对现代威胁，启用多因素认证可显著提升账户安全性。它通过组合“你知道的”、“你拥有的”和“你是什么的”三类因素，有效防止凭证泄露后的未授权访问。
数据加密是保护信息机密性的基石，分为静态加密和传输加密。对于敏感数据，无论存储在硬盘还是数据库中，都应进行加密处理。传输过程中

In [12]:
client = OpenAI(
    api_key=DEEPSEEK_API_KEY,
    base_url="https://api.deepseek.com"
)


def ask_deepseek(prompt):
    response = client.chat.completions.create(
        model="deepseek-chat",
        messages=[
            {
                "role": "system",
                "content": "你是一个严谨的知识库问答助手，只能根据用户提供的检索资料回答。"
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.2
    )
    
    return response.choices[0].message.content

In [13]:
test_response = client.chat.completions.create(
    model="deepseek-chat",
    messages=[
        {"role": "user", "content": "请只回复：API测试成功"}
    ],
    temperature=0.2
)

print(test_response.choices[0].message.content)

API测试成功


In [14]:
def rag_answer(query, top_k=3, threshold=0.35, show_context=True):
    """
    完整 RAG 流程：
    1. 用户问题检索 Top-K chunks；
    2. 展示检索结果；
    3. 如果最高相似度低于阈值，直接返回信息不足；
    4. 否则拼接 Prompt，调用 DeepSeek 生成回答。
    """
    retrieved_docs = retrieve(query, top_k=top_k)
    
    if show_context:
        show_retrieved_docs(retrieved_docs)
    
    best_score = retrieved_docs[0]["score"] if retrieved_docs else 0
    
    if best_score < threshold:
        return {
            "query": query,
            "answer": "知识库中没有足够信息回答该问题。",
            "retrieved_docs": retrieved_docs,
            "best_score": best_score,
            "status": "insufficient_context"
        }
    
    prompt = build_prompt(query, retrieved_docs)
    answer = ask_deepseek(prompt)
    
    return {
        "query": query,
        "answer": answer,
        "retrieved_docs": retrieved_docs,
        "best_score": best_score,
        "status": "answered"
    }

In [15]:
query = "中间人攻击是什么？"

result = rag_answer(
    query=query,
    top_k=3,
    threshold=0.35,
    show_context=True
)

print("\n========== 模型回答 ==========")
print(result["answer"])

print("\n========== 状态信息 ==========")
print("status:", result["status"])
print("best_score:", result["best_score"])


========== 检索结果 ==========

Top 1
score: 0.5461
source: 02_常见网络攻击类型.txt
chunk_id: 0
chunk_length: 378
text:
常见网络攻击类型
在网络安全领域，了解攻击者的手段是构建防御体系的基础。社会工程学是最具欺骗性的攻击方式，攻击者利用人性弱点如好奇心或恐惧心理，伪装成合法实体诱骗受害者泄露敏感信息。典型的钓鱼邮件会伪造发件人地址，诱导用户点击恶意链接。
分布式拒绝服务攻击旨在耗尽目标服务器资源，导致其无法为正常用户提供服务。攻击者通常控制大量被感染的“僵尸”计算机，同时向目标发起海量请求，造成带宽拥堵或系统崩溃。
中间人攻击发生在通信双方不知情的情况下，攻击者秘密中继并可能篡改双方的通信内容。在不安全的公共Wi-Fi环境中，攻击者可以轻易实施此攻击，窃取用户的登录凭证。
SQL注入攻击针对数据库驱动的应用程序
--------------------------------------------------------------------------------

Top 2
score: 0.3464
source: 03_安全防护与最佳实践.txt
chunk_id: 0
chunk_length: 359
text:
网络安全防护与最佳实践
构建有效的网络安全防护体系需要遵循纵深防御原则，即通过部署多层安全控制措施，确保单一防线被突破后仍有其他机制提供保护。
身份认证是安全的第一道关卡。仅依赖密码已不足以应对现代威胁，启用多因素认证可显著提升账户安全性。它通过组合“你知道的”、“你拥有的”和“你是什么的”三类因素，有效防止凭证泄露后的未授权访问。
数据加密是保护信息机密性的基石，分为静态加密和传输加密。对于敏感数据，无论存储在硬盘还是数据库中，都应进行加密处理。传输过程中必须使用TLS/SSL协议，这在用户访问网站时应体现为地址栏的锁形图标。
漏洞管理同样关键。软件供应商会发布补丁修复已知漏洞，而攻击者常利
--------------------------------------------------------------------------------

Top 3
score: 0.3401
source: 01_网络安全基础概念.txt


In [16]:
query = "重庆今天的天气怎么样？"

result = rag_answer(
    query=query,
    top_k=3,
    threshold=0.35,
    show_context=True
)

print("\n========== 模型回答 ==========")
print(result["answer"])

print("\n========== 状态信息 ==========")
print("status:", result["status"])
print("best_score:", result["best_score"])


========== 检索结果 ==========

Top 1
score: 0.0095
source: 01_网络安全基础概念.txt
chunk_id: 1
chunk_length: 80
text:
除了CIA三原则，可审计性和不可抵赖性也是现代网络安全的重要延伸属性。防火墙作为第一道防线，按照既定规则过滤进出网络的数据包，是网络安全架构中最基础的设施之一。
--------------------------------------------------------------------------------

Top 2
score: -0.0521
source: 01_网络安全基础概念.txt
chunk_id: 0
chunk_length: 357
text:
网络安全基础概念
网络安全是指保护网络系统中的硬件、软件及数据资源，防止因偶然或恶意原因遭到破坏、更改或泄露，确保系统连续可靠地运行。
网络安全的核心目标遵循CIA三元组原则：
1. 机密性：确保信息不泄露给非授权的个人或实体。常见的实现技术包括数据加密、访问控制列表等。例如，在传输用户密码时使用HTTPS协议进行加密，就是保障机密性的典型应用。
2. 完整性：保护信息不被未授权的篡改或破坏。哈希校验和数字签名是主要技术手段。当你从官方网站下载软件时，通常会提供一个MD5或SHA-256校验值，用户可以据此验证文件是否被植入恶意代码。
3. 可用性：保证被授权实体在需要时可以访问和使用相关资源
--------------------------------------------------------------------------------

Top 3
score: -0.0717
source: 02_常见网络攻击类型.txt
chunk_id: 0
chunk_length: 378
text:
常见网络攻击类型
在网络安全领域，了解攻击者的手段是构建防御体系的基础。社会工程学是最具欺骗性的攻击方式，攻击者利用人性弱点如好奇心或恐惧心理，伪装成合法实体诱骗受害者泄露敏感信息。典型的钓鱼邮件会伪造发件人地址，诱导用户点击恶意链接。
分布式拒绝服务攻击旨在耗尽目标服务器资源，导致其无法为正常用户提供服务。攻击者通常控制大量被感染的“僵尸”计算机，同时向目标发起海

In [17]:
os.makedirs("index", exist_ok=True)

faiss.write_index(index, "index/faiss.index")

with open("index/chunks.json", "w", encoding="utf-8") as f:
    json.dump(all_chunks, f, ensure_ascii=False, indent=2)

print("FAISS 索引和 chunks.json 已保存到 index 文件夹。")

FAISS 索引和 chunks.json 已保存到 index 文件夹。


In [18]:
def keyword_hit(answer, expected_keywords):
    """
    检查回答中是否命中预期关键词。
    """
    if not expected_keywords or pd.isna(expected_keywords):
        return None
    
    keywords = [kw.strip() for kw in expected_keywords.split(";") if kw.strip()]
    
    if not keywords:
        return None
    
    hit_keywords = [kw for kw in keywords if kw in answer]
    hit_rate = len(hit_keywords) / len(keywords)
    
    return {
        "keywords": keywords,
        "hit_keywords": hit_keywords,
        "hit_rate": hit_rate
    }


def source_hit(retrieved_docs, expected_source):
    """
    检查 Top-K 检索结果中是否包含期望来源。
    """
    if expected_source == "none" or pd.isna(expected_source):
        return None
    
    retrieved_sources = [doc["source"] for doc in retrieved_docs]
    
    return expected_source in retrieved_sources

In [19]:
eval_df = pd.read_csv("data/eval_questions.csv")

eval_results = []

for _, row in eval_df.iterrows():
    question = row["question"]
    expected_keywords = row["expected_keywords"]
    expected_source = row["expected_source"]
    should_answer = row["should_answer"]
    
    result = rag_answer(
        query=question,
        top_k=3,
        threshold=0.35,
        show_context=False
    )
    
    answer = result["answer"]
    status = result["status"]
    retrieved_docs = result["retrieved_docs"]
    
    kw_result = keyword_hit(answer, expected_keywords)
    src_hit = source_hit(retrieved_docs, expected_source)
    
    if should_answer == "yes":
        refusal_correct = status == "answered"
    else:
        refusal_correct = status == "insufficient_context"
    
    eval_results.append({
        "question": question,
        "should_answer": should_answer,
        "status": status,
        "best_score": result["best_score"],
        "expected_source": expected_source,
        "source_hit": src_hit,
        "keyword_hit_rate": kw_result["hit_rate"] if kw_result else None,
        "hit_keywords": ";".join(kw_result["hit_keywords"]) if kw_result else "",
        "refusal_correct": refusal_correct,
        "answer": answer
    })

eval_result_df = pd.DataFrame(eval_results)
eval_result_df

,question,should_answer,status,best_score,expected_source,source_hit,keyword_hit_rate,hit_keywords,refusal_correct,answer
0,什么是中间人攻击？,yes,answered,0.602234,02_常见网络攻击类型.txt,True,1.00,通信双方;秘密中继;篡改;公共Wi-Fi,True,根据检索资料，中间人攻击发生在通信双方不知情的情况下，攻击者秘密中继并可能篡改双方的通信内容...
1,SQL注入是什么？,yes,insufficient_context,0.156310,02_常见网络攻击类型.txt,True,0.00,,False,知识库中没有足够信息回答该问题。
2,什么是多因素认证？,yes,answered,0.351772,03_安全防护与最佳实践.txt,True,0.75,密码;拥有;账户安全,True,根据检索资料，多因素认证是一种身份认证方式，它通过组合“你知道的”（如密码）、“你拥有的”（...
3,防火墙的作用是什么？,yes,answered,0.457468,01_网络安全基础概念.txt,True,1.00,过滤;数据包;第一道防线,True,根据检索资料，防火墙的作用是：作为第一道防线，按照既定规则过滤进出网络的数据包，是网络安全架...
4,重庆今天的天气怎么样？,no,insufficient_context,0.009496,none,None,NaN,,True,知识库中没有足够信息回答该问题。
5,谁获得了今年诺贝尔奖？,no,insufficient_context,0.058264,none,None,NaN,,True,知识库中没有足够信息回答该问题。


In [20]:
os.makedirs("outputs", exist_ok=True)

eval_result_df.to_csv(
    "outputs/eval_results.csv",
    index=False,
    encoding="utf-8-sig"
)

print("评估结果已保存到 outputs/eval_results.csv")

评估结果已保存到 outputs/eval_results.csv


In [21]:
answerable_df = eval_result_df[eval_result_df["should_answer"] == "yes"]
unanswerable_df = eval_result_df[eval_result_df["should_answer"] == "no"]

print("可回答问题数量：", len(answerable_df))
print("不可回答问题数量：", len(unanswerable_df))

if len(answerable_df) > 0:
    print("平均关键词命中率：", answerable_df["keyword_hit_rate"].mean())
    print("来源命中率：", answerable_df["source_hit"].mean())

if len(unanswerable_df) > 0:
    print("拒答准确率：", unanswerable_df["refusal_correct"].mean())

可回答问题数量： 4
不可回答问题数量： 2
平均关键词命中率： 0.6875
来源命中率： 1.0
拒答准确率： 1.0
